# Zaman testleri — EKG modeli stresi mi, zamanı/cihazı mı öğreniyor? (GPU gerekmez, ~5 dk)

**Önce 05 not defteri en az 4. adıma kadar çalışmış olmalı** (Drive'da `bitkiEkg_sonuclar/asama1/gomme_*.npz` dosyaları).

**Çalışma zamanı → Tümünü çalıştır**, Drive iznini ver. GPU seçmene gerek yok; 05 açıkken başka sekmede çalıştırabilirsin.
Bitince bana "bitti" yaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
KOK = Path('/content/drive/MyDrive/bitkiEkg_sonuclar/asama1')
assert (KOK / 'gomme_ekg.npz').exists(), 'Önce 05 not defterinin 4. adımı bitmeli.'
!rm -rf /content/bitkiEkg
!git clone -q -b onDeneme --depth 1 https://github.com/melihakcam/bitkiEkg.git /content/bitkiEkg
import sys
sys.path.insert(0, '/content/bitkiEkg/src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from bitki_ekg.zaman_testleri import tum_testler

z = np.load('/content/bitkiEkg/data/colab/domates_ikili_6h_500.npz')
k = np.load('/content/bitkiEkg/data/colab/domates_kontrol_6h_500.npz')
son = np.concatenate([z['y'], k['y']]).astype(int)        # 0 = ilk 3 gün, 1 = son 3 gün
bitki = np.concatenate([z['bitki'], k['bitki']]).astype(int)

def sonda():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000, random_state=42))

bitki_satir, ozet_satir = [], []
for kol in ['ekg'] + [f'rastgele_t{t}' for t in range(1, 6)]:
    g = np.load(KOK / f'gomme_{kol}.npz')
    F = np.vstack([g['E'], g['Ec']])
    b, o = tum_testler(F, son, bitki, sonda)
    bitki_satir.append(b.assign(kol=kol)); ozet_satir.append(o.assign(kol=kol))
    print(kol, 'tamam', flush=True)
B, O = pd.concat(bitki_satir), pd.concat(ozet_satir)
B.to_csv(KOK / 'zaman_testleri_bitki.csv', index=False)
O.to_csv(KOK / 'zaman_testleri_ozet.csv', index=False)
pd.set_option('display.width', 200)
print(O[O.kol == 'ekg'].round(3).to_string(index=False))
O['k'] = np.where(O.kol == 'ekg', 'ekg', 'rastgele')
print(O.groupby(['test', 'k'])[['auc', 'p_perm']].mean().round(3).unstack(1))